# Analysis & reporting of preprocessed recordings

Select a single culture or a group of cultures (from the same experiment or across
experiments) and generate an MXtreme PDF report. 

Cultures are selected against the managed-store registry via `CultureID` / `CultureSelector`.

`mxtreme.analysis.generate_report` resolves the
selection, runs the requested sections, and writes one PDF.

This assumes the store already holds preprocessed `.npz` files and burst CSVs (see
`preprocess_pipeline.ipynb` and `burst_detection.ipynb`).

In [4]:
# Imports
from mxtreme.config import Config
from mxtreme.identity import CultureID, CultureSelector
from mxtreme.analysis import generate_report

### 1. Configure output directory 

MXtreme output is saved to `root_path` which can be fed directly when creating a `Config` object or can be read from `mxtreme.toml`. 

In this notebook, we will read cleaned npzs from the `preprocessed/` directory and burst data from the `burst_data/` directory and save output to the `analysis/` directory located in the `root_path`.


In [5]:
config = Config.from_toml("mxtreme.toml")
print("managed store:", config.data_root.resolve())
print("analysis dir: ", config.analysis_dir.resolve())

managed store: /Users/cgrass04/Code/mxtreme-dev/mxtreme-claude/claude_output
analysis dir:  /Users/cgrass04/Code/mxtreme-dev/mxtreme-claude/claude_output/analysis


### 2. Define the report sections

`sections` selects what goes in the PDF:

- `"overview"`    — single culture: ASDR + MEA layout per DIV; group: a concise summary table.
- `"activity"`    — firing rate / ISI / spike amplitude / active channels.
- `"bursting"`    — detection diagnostics + burst stats (IBI, rate, size, duration).
- `"stimulation"` — total stim/train time per DIV (skips non-stim cultures).
- `"performance"` — a learning curve from a user-defined objective (currently defaults to: burst direction).

The default selection is `("overview", "activity", "bursting")`.

In [6]:
SECTIONS = ("overview", "activity", "bursting", "stimulation", "performance")

### 3. Single culture analysis

A `CultureID(exp_id, chip, well)` expands to all completed DIVs for that culture. The overview
shows the ASDR and MEA layout.

In [ ]:
single = CultureID("burstTrainer", "M07140", "0")

report_path = generate_report(
    single, config,
    sections=SECTIONS,
)
report_path

## 2. A group from the same experiment

Two or more wells of the same MaxTwo chip. A `CultureSelector` yields population-level (mean ± SEM
across cultures) activity and burst panels, and the overview becomes a concise table.

In [ ]:
group_same_exp = CultureSelector(cultures=[
    CultureID("burstTrainer", "M07140", "0"),
    CultureID("burstTrainer", "M07140", "1"),
    CultureID("burstTrainer", "M07140", "2"),
])

generate_report(group_same_exp, config, sections=SECTIONS)

## 3. A group across experiments (MaxOne + MaxTwo)

Cultures from different experiments — here a MaxTwo well (`M07140`) and a MaxOne well
(`P004722`). `resolve_paths` groups them by experiment; the report pools them into one population
overview.

In [ ]:
group_cross_exp = CultureSelector(cultures=[
    CultureID("burstTrainer",   "M07140",   "0"),   # MaxTwo
    CultureID("m1BurstTrainer", "P004722", "0"),   # MaxOne
])

generate_report(group_cross_exp, config, sections=SECTIONS)

## Custom performance objective

The performance section's objective is pluggable: pass `objective_fn(burst_df) -> float`. The
default scores burst propagation direction (fraction with `origin_x < peak_x`). Supply your own for
a task-specific readout — e.g. mean burst size as a crude excitability proxy:

In [ ]:
def mean_size_objective(burst_df):
    return float(burst_df["size_frac_elec"].mean()) if len(burst_df) else float("nan")

generate_report(
    single, config,
    sections=("performance",),
    objective_fn=mean_size_objective,
    output_path=config.analysis_dir / "reports" / "M07140_well0_custom_objective.pdf",
)